三种微调方法：
    全量微调， 增量微调， 局部微调（使用with no_grad()或者其他的）
        模型预训练：从头开始训练一个模型（全新的模型为参数未更新的模型）
        微调训练（迁移学习）：基于训练好的模型继续学习新的任务：适应新的任务

        LoRA微调（使用LLaMA-Factory） Qlora：局部微调
                原理：拆为两个低秩矩阵与原始矩阵一起学习，训练后两权重合并
                ΔW = A @ B
                A: [512, r]
                B: [r, 768]
                W' = W + ΔW

                训练方法：sft
                lora缩放系数 = 2*lora的秩

                Qlora:降低参数精度到4bit来节约训练性能，合并时重建为16bit，训练不影响模型精度 启动qlora后：parameter:rank of lora:52 缩放系数：128 一般缩放系数为秩的二倍效果最好

        llamafactory生成的是lora矩阵不是原模型矩阵

        量化：float精度降低

        评价指标：bleu

        openwebui：python=3.11

    problem: 测试生成模型时decode之后的序列全为special token，将data【：，：-1】去除后仍无法解决：padding后去除末尾未去除特殊字符，解决方法：在生成过程中过滤special token
             在训练模型时collate function 中将data转换为列表传入tokenizer，训练时tokenizer将一个列表认为是一个token，训练精度极高测试结果均输出special token
             模型较大数据较小loss最开始会上升
             在平台上测试和本地部署后结果不一样？由于对话模板不同，模板影响数据集格式，模板对齐：将训练模板转换为jinjia格式 使用lmdeploy的话再将其转换为json格式
             虽然 LMDeploy 基础模板不直接支持工具调用，但通过 tool_spec 字段扩展了工具调用的起止标记，以兼容原模板中的 <tool_call> 和 <tool_response> 逻辑
                {
                    "model_name": "qwen",
                    "system": "<|im_start|>system\n",
                    "meta_instruction": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
                    "eosys": "<|im_end|>\n",
                    "user": "<|im_start|>user\n",
                    "eoh": "<|im_end|>\n",
                    "assistant": "<|im_start|>assistant\n",
                    "eoa": "<|im_end|>",
                    "separator": "\n",
                    "capability": "chat",
                    "stop_words": ["<|im_end|>", "</tool_call>", "<|im_start|>", "<|im_end|>\n"],
                    "tool_spec": {
                        "tool_call_start": "<tool_call>",
                        "tool_call_end": "</tool_call>",
                        "tool_response_start": "<tool_response>",
                        "tool_response_end": "</tool_response>"
                    }
            未收敛模型训练中断重新训练时loss上升：由于数据重新打乱后再次训练


llama-factory数据集制作
    单轮对话：
            "alpaca_zh_demo.json"
        {
        "instruction": "计算这些物品的总费用。 ",
        "input": "输入：汽车 - $3000，衣服 - $100，书 - $20。",
        "output": "汽车、衣服和书的总费用为 $3000 + $100 + $20 = $3120。"
        },

    第一次启动：
                [
        {
            "instruction": "人类指令（必填）",
            "input": "人类输入（选填）",
            "output": "模型回答（必填）",
            "system": "系统提示词（选填）",
            "history": [
            ["第一轮指令（选填）", "第一轮回答（选填）"],
            ["第二轮指令（选填）", "第二轮回答（选填）"]
            ]
        }
        ]

    多轮对话：
                [
        {
            "instruction": "今天的天气怎么样？",
            "input": "",
            "output": "今天的天气不错，是晴天。",
            "history": [
            [
                "今天会下雨吗？",
                "今天不会下雨，是个好天气。"
            ],
            [
                "今天适合出去玩吗？",
                "非常适合，空气质量很好。"
            ]
            ]
        }
        ]

分布式微调：
    由于显存限制单卡无法完整训练模型
    模型拆分使用不同显卡处理不同部分，可增加存储容量和训练速度
    多卡加载问题：
        怎么将单个模型的权重加载到不同卡上：
            数据并行：不拆分模型，拆分数据
            模型并行：拆分模型，记录模型深度（拆分与还原问题），要求显卡间通信
            流水线并行：将传播数据也进行拆分显存占用更小速度更慢
    deepspeed框架：多卡使用
        梯度检查点：减少激活值的显存占用，仅保留某些层的激活值，若需要其他值则当场计算
        cpu offloading：将优化器状态和梯度卸载到cpu内存中
        混合精度训练：FP16/BP16

        优势：显存占用低，易用性强

    llamafactory多卡：device account/deepspeed stage(2)

模型压缩：将大模型参数变少或存储变小，以减少其模型复杂度，减少模型复杂度，加快训练速度
    如量化，剪枝，知识蒸馏

混合精度：在训练时将参与训练的参数升为高精度，不参与训练的参数为低精度
    工业界最终选择了 INT8 量化—— FP32 在推理（inference）期间被 INT8 取代，而训练
    （training）仍然是 FP32。TensorRT，TensorFlow，PyTorch，MxNet 和许多其他深度学
    习软件都已启用（或正在启用）量化。
    通常，可以根据 FP32 和 INT8 的转换机制对解决方案进行分类。一些框架简单地引入了 
    Quantize 和 Dequantize 层，当从卷积或全链接层送入或取出时，它将 FP32 转换为 
    INT8 或相反。在这种情况下，如图四的上半部分所示，模型本身和输入/输出采用 FP32 
    格式。深度学习框架加载模型，重写网络以插入Quantize 和 Dequantize 层，并将权重转
    换为 INT8 格式。
    计算时：输入为32位转换为8位经过神经网络时反量化为32加权时转换为8位
    一般在qlora中：lora adapter中的参数和优化器的参数会被保存为高精度（fp16、bp16），原模型会被保存为低精度（4bit）
剪枝：减少非核心参数，若剪到模型核心参数则
知识蒸馏：有训练好的模型且该模型效果较好，使用两个损失一个是原本的损失，另一个是与教师网络的差异损失，且对teacher损失的依赖值逐渐降低

大模型分布式推理
    张量并行：将权重切割到不同的gpu上
    解决：单卡显存不足，高并发请求，用于加速推理

    vllm分布式机制：
    张量并行
    pageattention：解决模型分块问题，对transformer做优化

    LMDeploy分布式机制：
    张量并行，KV Cache量化：在推理过程中量化，动态显存管理

    部署方案：
        模型并行，transformer优化，动态窗口，pageattention

    lmdeploy： 
        轻量化4bit（awq）：离线
             8bit k/v：在线
        推理引擎：turbomind， pytorch
        k/v cache：将每次生成的attention的k v值保存不必每次均计算

        lmdeploy在线量化：lmdeploy serve api_server internlm/internlm2_5-7b-chat --quant-policy 8
        lmdeploy转换turbomind框架： 
                                在线转换：
                                    lmdeploy serve api_server D:\python_d\dl\qwen\models--Qwen--Qwen2.5-1.5B-Instruct\snapshots\989aa7980e4cf806f80c7fef2b1adb7bc71aa306 --model-name qwen
                                    lmdeploy chat qwen
                                离线转换：
                                    lmdeploy convert Qwen2.5-1.5B-Instruct(模型名称) D:\python_d\dl\qwen\models--Qwen--Qwen2.5-1.5B-Instruct 
    大模型是访存密集型任务：
    大语言模型推理，尤其是自回归生成（一边生成一边喂回模型）的时候：
        需要频繁访问 KV Cache（注意力键值缓存）。KV Cache 通常是很大的（因为 sequence 很长）。每次生成一个新 token，要去读之前所有 token 的 KV。
        这些操作主要是内存读，而不是算很多复杂的数学。
        所以：
        读 KV → 更新 KV → 读 KV → 生成新 Token这过程远远慢于显卡的算力，导致显卡算力吃不满。
        解决方法：只量化权重，加速且降低显存同时达到算力及显存瓶颈

        k/v cache：将历史对话存储不用重新计算上一轮对话



大模型评估：
    客观评估
        准确率，困惑度（ppl），生成质量（使用生成的数据集评价），rouge，条件对数概率（适用于复杂推理任务）
    opencompass：
        测试维度：知识类，推理类，语言类，代码类，多模态分析
        做特定数据集的评估